Phase 5 — The Evaluation Harness
=================================

Four metrics, in the RAGAS tradition, with one deliberate departure and
one thing added that RAGAS does not provide.

  context_precision   how much of the retrieved context was relevant
  context_recall      how much of the needed evidence was retrieved
  faithfulness        are the answer's claims supported by the context
  answer_relevancy    does the answer address the question

THE DEPARTURE. RAGAS computes all four with an LLM, because it assumes no
ground-truth labels — a reasonable default for the common case. This
project HAS exact labels: every golden question names its source
articles. So the first two are computed exactly, not judged. Using a
model to estimate a quantity you can compute is worse on every axis:
noisier, priced per question, not reproducible across model versions, and
it injects the judge's error into a number that had none.

THE ADDITION. The judge is validated before any of its scores are
believed. The standard practice — report faithfulness to three decimals
from a model whose agreement with ground truth was never measured — hides
an unknown error rate inside the headline number. Section B measures the
judge against cases where the right verdict is known by construction, and
Section C only reports real scores if it passed.

Sections
--------
  A. What is running
  B. Validating the judge BEFORE trusting it
  C. Context metrics — exact, and read against their ceiling
  D. Judged answer quality
  E. Failure attribution — retrieval or generation
  F. The LLM arms, judged (needs a credential and a judge that passed B)
  G. Verdict and handoff to Phase 6

In [ ]:
import importlib.util
import os
import sys
from pathlib import Path

# Locate the project root without assuming where Jupyter was launched.
#
# Two wrong assumptions were made here before, each breaking a real
# workflow:
#
#   Path.cwd().parent   assumes the kernel started in notebooks/. Launch
#                       Jupyter from the project root and it resolves one
#                       level too high.
#   walking up only     assumes the kernel started at or below the
#                       project. Launch Jupyter from a folder holding
#                       several projects — a normal thing to do — and the
#                       project is BELOW cwd, so upward search never
#                       finds it.
#
# So: search up, then search down. The marker is a file unique to this
# project, not a generic name like src/ that a sibling project might
# also have.
_MARKER = Path("src") / "corpus" / "seed_articles.py"


def _find_project_root(start: Path) -> Path:
    start = start.resolve()

    for candidate in [start, *start.parents]:          # up
        if (candidate / _MARKER).is_file():
            return candidate

    for pattern in ("*", "*/*"):                       # then down, bounded
        found = sorted({m.parents[2]
                        for m in start.glob(f"{pattern}/{_MARKER.as_posix()}")})
        if len(found) == 1:
            return found[0]
        if len(found) > 1:
            names = ", ".join(p.name for p in found)
            raise RuntimeError(
                f"Found several candidate project roots below {start}: "
                f"{names}. Start Jupyter inside the one you want."
            )

    raise RuntimeError(
        f"Could not locate the project root from {start}. Expected to find "
        f"{_MARKER.as_posix()} at, above, or within two levels below it."
    )


PROJECT_ROOT = _find_project_root(Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT))
import os
os.chdir(PROJECT_ROOT)          # so data/ and reports/ paths resolve

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.corpus.build import load_corpus, load_golden_set
from src.evaluation.judge import LexicalJudge, get_judge
from src.evaluation.judge_validation import VALIDATION_CASES, validate_judge
from src.evaluation.judged_arms import (
    contradiction_stance,
    estimate_usd,
    paired_mean_difference,
    parse_arms,
    plan_judging,
    report_to_dict,
    wilson_lower_bound,
    write_metrics,
)
from src.evaluation.rag_metrics import (
    aggregate_run,
    attribution_table,
    context_precision_ceiling,
    evaluate_run,
)
from src.generation.extractive import ExtractiveAnswerer
from src.generation.pipeline import LLMAnswerer, RAGPipeline
from src.generation.prompts import VARIANTS
from src.retrieval.chunking import STRATEGIES
from src.retrieval.retrievers import BM25Retriever

FIG_DIR = Path("reports/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR = Path("reports/metrics")

STRATEGY = "markdown_section"      # locked in by Phase 3
DEPTH = 15
EXTRACTIVE_THRESHOLD = 0.35        # tuned on dev in Phase 4

articles = load_corpus()
golden = load_golden_set()
references = {q["question_id"]: q.get("reference_answer", "") for q in golden}
chunks = STRATEGIES[STRATEGY](articles)
retriever = BM25Retriever(chunks)

chunks_per_article: dict[str, int] = {}
for c in chunks:
    chunks_per_article[c.article_id] = chunks_per_article.get(c.article_id, 0) + 1

## A. What is running

In [ ]:
print("=" * 78)
print("A. SETUP")
print("=" * 78)

# See notebook 04: a credential alone is not enough to make a call, the
# SDK has to be importable too. provider_ready() checks both and returns
# a reason naming the missing piece.
from src.llm.provider import provider_ready

has_key, blocker = provider_ready()
_sdk_found = importlib.util.find_spec("openai") is not None

if has_key:
    from src.llm.provider import get_provider
    provider = get_provider()
    judge = get_judge(provider, model=os.getenv("JUDGE_MODEL", "gpt-4o"))
else:
    judge = LexicalJudge()

n_refs = sum(1 for q in golden if q.get("reference_answer"))
judge_model = os.getenv("JUDGE_MODEL", "gpt-4o")

# Which LLM arms Section F judges, and what the whole run costs. Worked
# out here, before the first call, so a run is never started blind.
arm_names = parse_arms(os.getenv("JUDGE_ARMS"), VARIANTS)
plan = plan_judging(len(golden), n_refs, len(VALIDATION_CASES), len(arm_names))
max_calls = int(os.getenv("MAX_LLM_CALLS_PER_RUN", "2000"))
# Generation is normally served from the Phase 4 cache; if the prompts
# changed it is not, and those calls share the same ceiling.
worst_case_calls = plan.total_calls + len(golden) * len(arm_names)

print(f"""
  Retrieval        : {STRATEGY} + BM25 @ depth {DEPTH}
  Questions        : {len(golden)}
  Judge            : {judge!r}
  Credential       : {has_key}
""")

if has_key:
    # gpt-4o list pricing. Stated because the judge tier is deliberately
    # the EXPENSIVE model — the whole point of the asymmetry — so this
    # notebook costs roughly eight times what the generation notebook
    # does despite making fewer calls.
    arms_label = ", ".join(arm_names)
    print(f"""  Judge calls and cost, at {judge_model} list pricing. An estimate from
  average prompt sizes, not a quote:

    validation suite (Section B)        : {plan.validation_calls:>5} calls   ~${estimate_usd(plan.validation_calls):.2f}
    extractive baseline (Sections C-E)  : {plan.baseline_calls:>5} calls   ~${estimate_usd(plan.baseline_calls):.2f}
    LLM arms (Section F): {arms_label:<14}: {plan.arm_calls:>5} calls   ~${estimate_usd(plan.arm_calls):.2f}
    total                               : {plan.total_calls:>5} calls   ~${plan.total_usd:.2f}

  Generating the LLM answers is served from the Phase 4 cache when the
  prompts are unchanged, and costs cents on the generation model when
  they are not. Every response is cached on disk, so re-running this
  notebook costs nothing.

  The ordering is the cost control. Section B runs first and costs
  cents. If the judge fails validation there, Section F does not run:
  the ~${estimate_usd(plan.arm_calls):.2f} it would cost would buy numbers from an instrument already
  known to be miscalibrated. The gate is scientific first and economical
  second, but it is both.
""")
    if max_calls > 0 and worst_case_calls > max_calls:
        raise SystemExit(
            f"This run needs up to {worst_case_calls} LLM calls and "
            f"MAX_LLM_CALLS_PER_RUN is {max_calls}. The ceiling raises "
            f"mid-loop, after the first arms are already paid for, so the "
            f"run stops here instead. Raise MAX_LLM_CALLS_PER_RUN in .env "
            f"or judge fewer arms with JUDGE_ARMS."
        )

if not has_key:
    print(f"""  LLM judge unavailable.

  Reason: {blocker}

  So the LEXICAL judge runs instead. That is
  not a mock — it scores by term overlap, which is a real (weak) method,
  and Section B measures exactly how weak.

  The context metrics in Section C are UNAFFECTED by this: they are
  computed from ground-truth labels and need no model at all. That is
  the practical payoff of not delegating them to a judge — a credential
  outage degrades the evaluation instead of stopping it.

  With a key, the judge would be {os.getenv('JUDGE_MODEL', 'gpt-4o')} grading
  {os.getenv('GENERATION_MODEL', 'gpt-4o-mini')} output. The asymmetry is deliberate: a model
  grading its own output family shows self-preference bias, which would
  inflate the exact metric this project is built around.

  If that reason looks wrong — you installed the package, or
  `python -m src.llm.provider --check` reports READY in a terminal —
  then the KERNEL is running a different Python than the terminal. That
  is the usual cause, and these two lines identify it:

    kernel Python : {sys.executable}
    openai found  : {_sdk_found}

  Compare that path against `where python`. If they differ, either
  install into the kernel's own environment from a notebook cell:

      %pip install openai

  or point the notebook at the interpreter you already installed into.
  Restart the kernel afterwards — pip does not refresh an already
  running interpreter's import state.
""")

## B. Validating the judge

In [ ]:
print("\n" + "=" * 78)
print("B. VALIDATING THE JUDGE — BEFORE TRUSTING ANY SCORE")
print("=" * 78)
print(f"""
An LLM judge is a measuring instrument. An uncalibrated instrument
produces numbers, not measurements.

{len(VALIDATION_CASES)} cases where the correct verdict follows from how the case was
written, not from an opinion:

  supported     every claim appears in the context
  fabricated    a plausible fact the context never states
  contradicted  a fact that directly opposes the context
  refusal       no claims made at all
  off_topic     faithful to the context, answers a different question

The most diagnostic is `contradicted`. A contradicting sentence reuses
the context's vocabulary almost perfectly — high overlap, opposite
meaning — so it is the case that separates semantic judgement from term
matching.
""")

report = validate_judge(judge)

print(f"  Judge: {report.judge}\n")
print(f"    faithfulness accuracy : {report.faithfulness_accuracy:>6.1%}")
print(f"    relevancy accuracy    : {report.relevancy_accuracy:>6.1%}")
print(f"    overall               : {report.overall_accuracy:>6.1%}")
print(f"    parse failures        : {report.parse_failures}")
print(f"    TRUSTWORTHY           : {report.is_trustworthy}")

print("\n  Accuracy by case kind:\n")
for kind, acc in report.accuracy_by_kind().items():
    bar = "#" * int(acc * 20)
    print(f"    {kind:<14} {acc:>5.0%}  {bar}")

fails = report.failures()
if fails:
    print(f"\n  Failures ({len(fails)}):\n")
    for r in fails:
        want_f = "high" if r.case.expect_faithful_high else "LOW"
        want_r = "high" if r.case.expect_relevant_high else "LOW"
        print(f"    [{r.case.case_id}] {r.case.kind}")
        print(f"      faithfulness {r.faithfulness.score:.2f} (want {want_f})"
              f"   relevancy {r.relevancy.score:.2f} (want {want_r})")
        if r.faithfulness.n_claims is not None:
            # The ratio alone hides WHY a case failed: a judge that
            # found the bad claim but split the answer into more claims
            # than expected lands above the threshold for a different
            # reason than one that missed the bad claim entirely.
            print(f"      claims supported {r.faithfulness.n_supported}"
                  f" of {r.faithfulness.n_claims}")
        if r.case.note:
            print(f"      {r.case.note}")

if not report.is_trustworthy:
    print(f"""
>>> THE JUDGE FAILS VALIDATION. Its scores in Sections D and E are
    reported as a demonstration that the harness runs, NOT as evidence
    about answer quality.

  This is the correct outcome for the lexical judge and the reason it is
  in the repo. Look at where it fails:

    contradicted   it scores contradictions as fully faithful, because a
                   contradicting sentence shares nearly every term with
                   the context it contradicts. Term overlap cannot
                   represent negation or numeric disagreement.
    refusal        it scores a refusal as maximally unfaithful, because a
                   refusal shares no vocabulary with the context. But a
                   refusal asserts nothing, so it is vacuously faithful.

  Those two failures are not fixable by tuning a threshold — they are
  what "semantic" means. This is the concrete argument for paying for an
  LLM judge, made by measurement rather than assertion.

  A gate is what makes this useful: if the real judge fails this suite
  too, its numbers should be discarded rather than published.
""")
else:
    print(f"""
  The judge passes at {report.overall_accuracy:.0%}. Its scores below carry that
  known error rate — which is the point of measuring it. A judge is
  never perfect; an unmeasured judge is merely a judge whose error rate
  is unknown.
""")

## C. Context metrics

In [ ]:
print("\n" + "=" * 78)
print("C. CONTEXT METRICS — EXACT, NO JUDGE INVOLVED")
print("=" * 78)

pipeline = RAGPipeline(retriever,
                       ExtractiveAnswerer(min_score=EXTRACTIVE_THRESHOLD),
                       depth=DEPTH)
results = pipeline.run(golden)
evaluated = evaluate_run(results, judge=judge, references=references)
report_run = aggregate_run(evaluated)

in_scope_ev = [e for e in evaluated if not e.result.is_out_of_scope]
ceilings = [context_precision_ceiling(e.result, chunks_per_article)
            for e in in_scope_ev]
ceilings = [c for c in ceilings if c is not None]
mean_ceiling = float(np.mean(ceilings))

mean_chunks = float(np.mean(list(chunks_per_article.values())))
single_gt = [e for e in in_scope_ev if len(e.result.gt_article_ids) == 1]
single_ceiling = float(np.mean(
    [context_precision_ceiling(e.result, chunks_per_article) for e in single_gt]))

print(f"""
  context_recall    : {report_run.context_recall:.3f}
  context_precision : {report_run.context_precision:.3f}

>>> The precision number is meaningless without its ceiling.

  Mean ACHIEVABLE context_precision at depth {DEPTH}: {mean_ceiling:.3f}

  The ceiling is arithmetic, not a modelling choice. Articles split into
  ~{mean_chunks:.1f} chunks on this strategy, so a question whose answer lives in
  ONE article has at most ~{mean_chunks:.1f} relevant chunks in existence. Retrieved
  at depth {DEPTH}, that caps it near {single_ceiling:.2f} — the other slots MUST hold
  something irrelevant. Questions needing several articles have
  proportionally more relevant material available, which is why the
  mean ceiling ({mean_ceiling:.3f}) sits above the single-article one ({single_ceiling:.3f}).

  So {report_run.context_precision:.3f} against {mean_ceiling:.3f} means the retriever achieves {report_run.context_precision / mean_ceiling:.0%} of
  what is structurally possible. Read against an implicit ceiling of 1.0
  — which is how this metric is usually reported — the same system looks
  broken.

  The ceiling is the bill for Phase 3's choices arriving: depth {DEPTH} buys
  recall {report_run.context_recall:.3f} and pays with a context window that is ~{1 - report_run.context_precision:.0%}
  irrelevant text. That trade is what Phase 6 has to weigh, because
  distractor density is what drives a generator to invent things.
""")

by_cat = {}
for cat in ("single_hop", "multi_hop", "ambiguous"):
    subset = [e for e in in_scope_ev if e.category == cat]
    if subset:
        ceil = float(np.mean([context_precision_ceiling(e.result,
                                                        chunks_per_article)
                              for e in subset]))
        prec = float(np.mean([e.context_precision for e in subset]))
        by_cat[cat] = (float(np.mean([e.context_recall for e in subset])),
                       prec, ceil)

print(f"  {'category':<14}{'recall':>9}{'precision':>11}{'ceiling':>10}{'% of max':>10}")
print("  " + "-" * 54)
for cat, (rec, prec, ceil) in by_cat.items():
    print(f"  {cat:<14}{rec:>9.3f}{prec:>11.3f}{ceil:>10.3f}{prec / ceil:>10.0%}")

worst = min(by_cat, key=lambda c: by_cat[c][1] / by_cat[c][2])
print(f"""
  Reading the columns together inverts the story the raw numbers tell.

  single_hop looks WORST on raw precision ({by_cat['single_hop'][1]:.3f}) but is nearest its
  ceiling ({by_cat['single_hop'][1] / by_cat['single_hop'][2]:.0%}). It scores low because one article cannot fill {DEPTH}
  slots — not because retrieval is failing. multi_hop's higher precision
  ({by_cat['multi_hop'][1]:.3f}) is mostly its higher ceiling ({by_cat['multi_hop'][2]:.3f}), not better retrieval.

>>> And the real weak spot is {worst}, at {by_cat[worst][1] / by_cat[worst][2]:.0%} of its ceiling.

  That category has the MOST relevant material available of the three
  ({by_cat[worst][2]:.3f} ceiling, because these questions map to several articles)
  and finds the LEAST of it. Raw precision hid this completely —
  {worst} and single_hop look almost identical at {by_cat[worst][1]:.3f} vs {by_cat['single_hop'][1]:.3f},
  while one is near its limit and the other is nowhere near.

  This sharpens the open item Phase 3 left. Ambiguous questions were the
  weak category on recall too, and the pattern across both metrics says
  the same thing: the system is not failing to rank the right articles,
  it is failing to work out WHICH articles an underspecified question is
  about. That is a query-understanding problem, and no amount of
  retrieval or prompt tuning addresses it.
""")

## D. Judged answer quality

In [ ]:
print("\n" + "=" * 78)
print("D. JUDGED ANSWER QUALITY")
print("=" * 78)

if not report.is_trustworthy:
    print("""
  Reported for completeness. The judge failed validation, so these are
  numbers the harness produced, not evidence about the answers. Treating
  them as findings would be the exact error Section B exists to prevent.
""")

print(f"""
  answerer                       : {report_run.answerer}
  faithfulness (answered only)   : {report_run.faithfulness:.3f}   <- headline
  faithfulness (all questions)   : {report_run.faithfulness_all:.3f}
  relevancy                      : {report_run.relevancy:.3f}
  correctness                    : {report_run.correctness:.3f}

  answered {report_run.n_answered} of {report_run.n} questions; the rest were refusals.

>>> Faithfulness is reported over ANSWERED questions only, and the gap
    between the two lines above is why.

  A refusal asserts nothing, so a correct judge scores it vacuously
  faithful at 1.0. Averaging refusals in therefore rewards a system for
  declining to be useful — and taken to its limit, a system that refuses
  every question reports PERFECT faithfulness while answering nothing.

  The answered-only figure means what people think faithfulness means:
  when this system does make claims, how often are they supported. The
  all-questions figure is kept visible rather than deleted so the
  inflation is auditable rather than hidden.
""")

if report_run.faithfulness_all < report_run.faithfulness:
    print("""  (Here the two run in the opposite direction, because this judge
  scores refusals near ZERO rather than 1.0 — one of the failures
  Section B catches. With a judge that handles refusals correctly, the
  all-questions number would sit ABOVE the answered-only one.)
""")
else:
    print(f"""  (Here the inflation is visible: {report_run.faithfulness_all:.3f} over all questions against
  {report_run.faithfulness:.3f} over answered ones. The refusals are being scored as
  vacuously faithful, which is correct for each refusal and misleading
  as an average.)
""")

print("  Failure modes across all 120 questions:\n")
for mode, count in report_run.failure_modes.items():
    print(f"    {mode:<24} {count:>4}  ({count / len(evaluated):>5.1%})")

print("""
  These categories are the deliverable, more than the scores are. A
  single "hallucination rate" would collapse generation_failure,
  retrieval_failure and over_refusal into one number — and those three
  have completely different fixes.
""")

## E. Failure attribution

In [ ]:
print("=" * 78)
print("E. FAILURE ATTRIBUTION — RETRIEVAL OR GENERATION?")
print("=" * 78)
print("""
The thread from Phase 3 arrives here. Cross-tabulating faithfulness
against whether the evidence was retrieved at all separates two failures
that look identical in the output and need opposite fixes.
""")

table = attribution_table(evaluated)
total = sum(table.values())
print(f"  {'bucket':<26}{'n':>5}{'share':>9}   interpretation")
print("  " + "-" * 78)
interp = {
    "evidence+faithful": "working as intended",
    "evidence+unfaithful": "GENERATION failure — evidence there, ignored",
    "no_evidence+faithful": "faithful to wrong context, or refused",
    "no_evidence+unfaithful": "RETRIEVAL failure — looks like hallucination",
    "refused": "no claims made; attribution not applicable",
    "out_of_scope": "scored on refusal in Phase 4, not here",
    "unjudged": "no judge available",
}
for bucket, n in table.items():
    print(f"  {bucket:<26}{n:>5}{n / total:>9.1%}   {interp[bucket]}")

gen_fail = table["evidence+unfaithful"]
ret_fail = table["no_evidence+unfaithful"]

if gen_fail + ret_fail == 0:
    print(f"""
>>> Zero unfaithful answers — and that is a statement about the PAIRING,
    not about quality.

  The extractive baseline builds answers by copying sentences out of the
  retrieved context verbatim. The lexical judge scores faithfulness by
  term overlap with that same context. A copier judged by overlap is
  faithful by construction: it is not passing a hard test, it is exempt
  from the test.

  This is the same category error as the extractive baseline's perfect
  citation precision in Phase 4 — a metric measuring the architecture
  rather than the behaviour. Reporting 1.000 faithfulness here as
  evidence the system is trustworthy would be exactly the mistake
  Section B's gate exists to catch.

  The attribution table only becomes informative when BOTH sides can
  fail: a generator that paraphrases (so it can drift from the source)
  and a judge that reads meaning (so it can notice). That needs an LLM
  on both ends, which is what a credential unlocks.

  The machinery is verified — the buckets populate, refusals and
  out-of-scope rows are correctly excluded — but the finding is pending.
""")
else:
    print(f"""
  Of the unfaithful answers, {gen_fail} had the evidence available and {ret_fail} did
  not. Only the first group is addressable by changing the generator.

  Sending all of them to a prompt-engineering fix — the reflex when a
  dashboard shows "hallucination rate: X%" — would aim {ret_fail} of them at
  the wrong component entirely.

  One thing this table shows that the lexical judge could not. This
  answerer copies sentences out of the context verbatim, and under term
  overlap that made it faithful by construction. A judge that reads
  meaning still finds {gen_fail} of its answers unfaithful — because it
  stitches together sentence FRAGMENTS from different articles, and a
  fragment cut mid-sentence and attached to another source's citation
  is no longer a claim that source makes. Copying words is not the same
  as preserving what they said.
""")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

kinds = list(report.accuracy_by_kind().keys())
accs = [report.accuracy_by_kind()[k] for k in kinds]
colors = ["#5AD8A6" if a >= 0.8 else "#F6735B" for a in accs]
ax1.barh(kinds, accs, color=colors, edgecolor="white")
ax1.axvline(0.8, linestyle="--", color="#2B2B2B", linewidth=1.2,
            label="trust threshold")
ax1.set_xlim(0, 1)
ax1.set_xlabel("Judge accuracy on known-label cases")
ax1.set_title(f"Is the judge trustworthy?\n{report.judge} — overall "
              f"{report.overall_accuracy:.0%}", fontweight="bold")
ax1.legend(fontsize=8, loc="lower right")
ax1.grid(axis="x", linestyle="--", alpha=0.4)
ax1.set_axisbelow(True)

cats = list(by_cat.keys())
x = np.arange(len(cats))
ax2.bar(x - 0.2, [by_cat[c][0] for c in cats], 0.4,
        label="context recall", color="#5B8FF9", edgecolor="white")
ax2.bar(x + 0.2, [by_cat[c][1] for c in cats], 0.4,
        label="context precision", color="#F6BD16", edgecolor="white")
ax2.axhline(mean_ceiling, linestyle="--", color="#2B2B2B", linewidth=1.2,
            label=f"precision ceiling ({mean_ceiling:.2f})")
ax2.set_xticks(x)
ax2.set_xticklabels(cats)
ax2.set_ylabel("Score")
ax2.set_title("Context quality by question type\nprecision is capped by "
              "chunk granularity", fontweight="bold")
ax2.legend(fontsize=8)
ax2.grid(axis="y", linestyle="--", alpha=0.4)
ax2.set_axisbelow(True)

plt.suptitle("Phase 5 — evaluation harness", fontsize=13,
             fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "05_evaluation.png", dpi=140, bbox_inches="tight",
            metadata={"Software": None})
print(f"  Saved -> {FIG_DIR}/05_evaluation.png")

## F. The LLM arms, judged

In [ ]:
print("\n" + "=" * 78)
print("F. THE LLM ARMS, JUDGED")
print("=" * 78)
print(f"""
Everything above scored the extractive baseline. That was the right
subject for building the harness — free, deterministic, failure modes
known in advance — and the wrong subject for a deployment decision,
because nobody proposes to ship the baseline.

An earlier version of this notebook stopped here even with a credential
present: the key swapped the lexical judge for an LLM judge and then
aimed it at the same extractive answers. The memo meanwhile described
the generation layer as one judge run away from being measured. That run
would have paid a model to grade a copier and left every LLM answer
unjudged, and nothing would have failed.

So this section does what the write-up always said was being done: it
runs the judge over LLM answers.

  Arms selected : {', '.join('llm_' + a for a in arm_names)}
  (set JUDGE_ARMS=all, or a comma-separated list, to change this)
""")

arm_evaluated: dict[str, list] = {}
arm_reports: dict[str, object] = {}

if not has_key:
    print("""  NOT MEASURED IN THIS RUN — no credential, so no LLM arm ran here and
  none was judged. The judged results from the credentialed run are
  recorded in reports/metrics/05_judged_arms.json, and Phase 7 checks
  the memo against that record.
""")
elif not report.is_trustworthy:
    print(f""">>> NOT RUN — the judge failed Section B at {report.overall_accuracy:.0%}.

  This is the gate doing its job. Judging the LLM arms now would cost
  ~${estimate_usd(plan.arm_calls):.2f} and produce numbers from an instrument already shown to
  be miscalibrated. They would look exactly like findings.

  Fix the judge first (a stronger JUDGE_MODEL, or a revised prompt
  re-validated against Section B), then re-run.
""")
else:
    for name in arm_names:
        answerer = LLMAnswerer(provider, VARIANTS[name])
        print(f"  {answerer.name}: generating over {len(golden)} questions ...",
              flush=True)
        arm_results = RAGPipeline(retriever, answerer, depth=DEPTH).run(
            golden, progress_every=40)
        print(f"  {answerer.name}: judging ...", flush=True)
        arm_evaluated[answerer.name] = evaluate_run(
            arm_results, judge=judge, references=references, progress_every=40)
        arm_reports[answerer.name] = aggregate_run(arm_evaluated[answerer.name])

if arm_reports:
    all_evaluated = {report_run.answerer: evaluated, **arm_evaluated}
    all_reports = {report_run.answerer: report_run, **arm_reports}

    def _parse_failures(evs) -> int:
        return sum(1 for e in evs
                   for j in (e.faithfulness, e.relevancy, e.correctness)
                   if j is not None and not j.parsed_ok)

    print(f"\n  {'answerer':<22}{'answered':>9}{'faithful':>10}{'relevant':>10}"
          f"{'correct':>9}{'gen fail':>10}{'ret fail':>10}{'ans OOS':>9}")
    print("  " + "-" * 89)
    for name, rep in all_reports.items():
        fm = rep.failure_modes
        print(f"  {name:<22}{rep.n_answered:>9}{rep.faithfulness:>10.3f}"
              f"{rep.relevancy:>10.3f}{rep.correctness:>9.3f}"
              f"{fm.get('generation_failure', 0):>10}"
              f"{fm.get('retrieval_failure', 0):>10}"
              f"{fm.get('answered_oos', 0):>9}")

    print("""
  How to read the columns:

    faithful   answered questions only (see Section D for why)
    correct    agreement with the reference answer, over ALL questions,
               so it includes the out-of-scope ones — where the reference
               says the corpus is silent and the right answer is a refusal
    gen fail   unfaithful although the evidence was retrieved
    ret fail   unfaithful and the evidence was NOT retrieved — an
               upstream failure that looks like a hallucination
    ans OOS    out-of-scope questions answered instead of refused
""")

    failures = {name: _parse_failures(evs) for name, evs in all_evaluated.items()}
    print("  Judge replies that could not be parsed (each is scored 0.0, so a")
    print("  large count here means the scores above are deflated, not low):\n")
    for name, n_bad in failures.items():
        print(f"    {name:<22} {n_bad:>4} of {len(all_evaluated[name]) * 3}")

    # --- By category ---------------------------------------------------
    cats_all = ["single_hop", "multi_hop", "ambiguous", "out_of_scope"]
    print("\n  Correctness by question type:\n")
    print(f"  {'answerer':<22}" + "".join(f"{c:>14}" for c in cats_all))
    print("  " + "-" * (22 + 14 * len(cats_all)))
    by_cat_correct: dict[str, dict[str, float]] = {}
    for name, evs in all_evaluated.items():
        row = {}
        for c in cats_all:
            vals = [e.correctness.score for e in evs
                    if e.category == c and e.correctness is not None]
            row[c] = float(np.mean(vals)) if vals else float("nan")
        by_cat_correct[name] = row
        print(f"  {name:<22}" + "".join(f"{row[c]:>14.3f}" for c in cats_all))

    # --- Paired comparison ---------------------------------------------
    comparisons = {}
    comparisons_split = {}
    arm_list = list(arm_evaluated)
    if len(arm_list) >= 2:
        print("""
  Paired differences in correctness. Both arms answer the same 120
  questions, so the comparison is per question — the shared difficulty
  cancels instead of widening the interval.
""")
        for i, a_name in enumerate(arm_list):
            for b_name in arm_list[i + 1:]:
                a_sc = [e.correctness.score for e in arm_evaluated[a_name]]
                b_sc = [e.correctness.score for e in arm_evaluated[b_name]]
                diff, lo, hi = paired_mean_difference(a_sc, b_sc)
                comparisons[f"{a_name} - {b_name}"] = (diff, lo, hi)
                verdict = ("interval excludes zero" if lo > 0 or hi < 0
                           else "interval includes zero — not distinguishable")
                print(f"    {a_name} - {b_name}: {diff:+.3f} "
                      f"[{lo:+.3f}, {hi:+.3f}]  {verdict}")
                # The all-questions figure nets two opposite effects: one
                # arm is more helpful on answerable questions, the other
                # is safer on unanswerable ones. Split, each is visible.
                for scope, keep in (("in-scope", False), ("out-of-scope", True)):
                    a_s = [e.correctness.score for e in arm_evaluated[a_name]
                           if e.result.is_out_of_scope == keep]
                    b_s = [e.correctness.score for e in arm_evaluated[b_name]
                           if e.result.is_out_of_scope == keep]
                    d2, lo2, hi2 = paired_mean_difference(a_s, b_s)
                    comparisons_split[f"{a_name} - {b_name} ({scope})"] = {
                        a_name: round(float(np.mean(a_s)), 6),
                        b_name: round(float(np.mean(b_s)), 6),
                        "difference": round(d2, 6),
                        "ci_low": round(lo2, 6), "ci_high": round(hi2, 6),
                        "n": len(a_s)}
                    print(f"      {scope:<13} {np.mean(a_s):.3f} vs {np.mean(b_s):.3f}   "
                          f"{d2:+.3f} [{lo2:+.3f}, {hi2:+.3f}]  (n={len(a_s)})")

    # --- The same questions, not the same averages ----------------------
    # The headline faithfulness figures above are NOT comparable across
    # arms: each is a mean over the questions THAT arm chose to answer. An
    # arm that answers everything is averaged over its worst cases; an arm
    # that refuses them is not. Restricting to questions both arms
    # answered removes that, and is the only like-for-like comparison.
    same_question = {}
    if len(arm_list) >= 2:
        print("""
  THE SAME QUESTIONS, NOT THE SAME AVERAGES. The faithfulness column
  above compares each arm on a different set of questions — the ones it
  chose to answer. Restricted to questions BOTH arms answered:
""")
        for i, a_name in enumerate(arm_list):
            for b_name in arm_list[i + 1:]:
                a_by = {e.question_id: e for e in arm_evaluated[a_name]}
                b_by = {e.question_id: e for e in arm_evaluated[b_name]}
                shared = [q for q in a_by
                          if not a_by[q].is_refusal and not b_by[q].is_refusal]
                if not shared:
                    continue
                fa = [a_by[q].faithfulness.score for q in shared]
                fb = [b_by[q].faithfulness.score for q in shared]
                ca = [a_by[q].correctness.score for q in shared]
                cb = [b_by[q].correctness.score for q in shared]
                wa = float(np.mean([len(a_by[q].result.answer.split()) for q in shared]))
                wb = float(np.mean([len(b_by[q].result.answer.split()) for q in shared]))
                fd = paired_mean_difference(fa, fb)
                cd = paired_mean_difference(ca, cb)
                same_question[f"{a_name} - {b_name}"] = {
                    "n_both_answered": len(shared),
                    "faithfulness": {a_name: round(float(np.mean(fa)), 6),
                                     b_name: round(float(np.mean(fb)), 6),
                                     "difference": round(fd[0], 6),
                                     "ci_low": round(fd[1], 6),
                                     "ci_high": round(fd[2], 6)},
                    "correctness": {a_name: round(float(np.mean(ca)), 6),
                                    b_name: round(float(np.mean(cb)), 6),
                                    "difference": round(cd[0], 6),
                                    "ci_low": round(cd[1], 6),
                                    "ci_high": round(cd[2], 6)},
                    "mean_answer_words": {a_name: round(wa, 1),
                                          b_name: round(wb, 1)},
                }
                print(f"    {a_name} vs {b_name}, n={len(shared)} questions both answered")
                print(f"      faithfulness  {np.mean(fa):.3f} vs {np.mean(fb):.3f}   "
                      f"difference {fd[0]:+.3f} [{fd[1]:+.3f}, {fd[2]:+.3f}]")
                print(f"      correctness   {np.mean(ca):.3f} vs {np.mean(cb):.3f}   "
                      f"difference {cd[0]:+.3f} [{cd[1]:+.3f}, {cd[2]:+.3f}]")
                print(f"      answer length {wa:.0f} vs {wb:.0f} words\n")
        print("""  Read the faithfulness line against the table above. Whatever gap
  the headline column shows between a refusing arm and a non-refusing
  one, this is how much of it survives on common ground. The rest was
  composition: one arm was being averaged over questions the other
  declined.
""")

    # --- What refusing costs, and what the safety number can bear -------
    n_oos = sum(1 for q in golden if q["category"] == "out_of_scope")
    refusal_detail = {}
    print("  REFUSALS. Where each arm declines, and whether it had the evidence:\n")
    for name, evs in arm_evaluated.items():
        in_scope_refused = [e for e in evs
                            if e.is_refusal and not e.result.is_out_of_scope]
        full = sum(1 for e in in_scope_refused if (e.context_recall or 0) == 1)
        part = sum(1 for e in in_scope_refused if 0 < (e.context_recall or 0) < 1)
        none = sum(1 for e in in_scope_refused if (e.context_recall or 0) == 0)
        by_category = {
            c: {"refused": sum(1 for e in in_scope_refused if e.category == c),
                "n": sum(1 for e in evs if e.category == c)}
            for c in ("single_hop", "multi_hop", "ambiguous")}
        oos_refused = sum(1 for e in evs
                          if e.result.is_out_of_scope and e.is_refusal)
        oos_answers = [e for e in evs
                       if e.result.is_out_of_scope and not e.is_refusal]
        oos_unsupported = sum(1 for e in oos_answers if e.faithfulness.score < 0.7)
        lower = wilson_lower_bound(oos_refused, n_oos)
        refusal_detail[name] = {
            "in_scope_refused": len(in_scope_refused),
            "in_scope_refused_with_full_evidence": full,
            "in_scope_refused_with_partial_evidence": part,
            "in_scope_refused_with_no_evidence": none,
            "in_scope_refused_by_category": by_category,
            "out_of_scope_refused": oos_refused,
            "out_of_scope_n": n_oos,
            "out_of_scope_refusal_wilson_lower_95": round(lower, 6),
            "out_of_scope_answers_judged_unsupported": oos_unsupported,
        }
        cats_txt = ", ".join(f"{c} {v['refused']}/{v['n']}"
                             for c, v in by_category.items())
        print(f"    {name}")
        print(f"      in-scope refused      : {len(in_scope_refused)}  "
              f"(evidence fully retrieved for {full}, partly for {part}, not at all for {none})")
        print(f"      by question type      : {cats_txt}")
        bound = (f"  -> true rate at least {lower:.1%} (95% Wilson lower bound)"
                 if oos_refused else "")
        print(f"      out-of-scope refused  : {oos_refused} of {n_oos}{bound}")
        if oos_answers:
            print(f"      out-of-scope answered : {len(oos_answers)}, of which "
                  f"{oos_unsupported} contain claims the judge found unsupported")
        print()
    print(f"""  Two readings that the raw counts invite and do not support.

  A clean sweep on {n_oos} questions is not a clean system. The Wilson bound
  is what {n_oos} questions can actually establish, and an underlying
  failure rate of one in ten would still produce a perfect score on a
  set this size about {0.9 ** n_oos:.0%} of the time.

  And "answered" is not "hallucinated". An arm with no refusal
  instruction never emits the canonical refusal, so every out-of-scope
  reply counts as answered — including the ones that say the context
  does not cover the question and then suggest something. The judged
  count beside it is the stricter statement: answers containing at
  least one claim the context does not support.
""")

    # --- The planted contradiction -------------------------------------
    print("""
  THE PLANTED CONTRADICTION (mh-011). Two retrieved articles state
  different refund windows: 14 days and 30 days. Phases 3 and 4 both
  deferred the question to here: does any arm surface the conflict, or
  does it assert one figure as though the other source did not exist?
""")
    mh011 = {}
    for name, evs in all_evaluated.items():
        e = next((x for x in evs if x.question_id == "mh-011"), None)
        if e is None:
            continue
        stance = contradiction_stance(e.result.answer)
        mh011[name] = {
            "stance": stance,
            "faithfulness": round(e.faithfulness.score, 6),
            "correctness": (round(e.correctness.score, 6)
                            if e.correctness is not None else None),
            "answer": e.result.answer,
        }
        corr = (f"{e.correctness.score:.2f}" if e.correctness is not None
                else "n/a")
        print(f"    {name}")
        print(f"      stance       : {stance}")
        print(f"      faithfulness : {e.faithfulness.score:.2f}   "
              f"correctness vs reference : {corr}")
        print(f"      answer       : {e.result.answer[:420]}\n")

    print("""  `states_both` means both figures appear, which is necessary for
  surfacing the conflict and not sufficient: an answer can list two
  windows without saying they disagree. The correctness score is against
  a reference answer that names the conflict explicitly, so the two
  columns are read together. Note what faithfulness CANNOT show here: an
  answer asserting "30 days" is fully faithful, because one retrieved
  article does say that. An answer can be fully faithful to one of two
  contradictory sources and still mislead the customer, and only this
  check and the reference comparison can see it.
""")

    # --- Record ---------------------------------------------------------
    record = {
        "generation_model": os.getenv("GENERATION_MODEL", "gpt-4o-mini"),
        "judge": repr(judge),
        "judge_validation": {
            "overall_accuracy": round(report.overall_accuracy, 6),
            "faithfulness_accuracy": round(report.faithfulness_accuracy, 6),
            "relevancy_accuracy": round(report.relevancy_accuracy, 6),
            "n_cases": len(VALIDATION_CASES),
            "parse_failures": report.parse_failures,
            "trustworthy": bool(report.is_trustworthy),
        },
        "retrieval": {"strategy": STRATEGY, "retriever": "bm25", "depth": DEPTH},
        "n_questions": len(golden),
        "runs": {name: report_to_dict(rep) for name, rep in all_reports.items()},
        "judge_parse_failures": failures,
        "correctness_by_category": {
            name: {c: (None if v != v else round(v, 6)) for c, v in row.items()}
            for name, row in by_cat_correct.items()},
        "paired_correctness": {
            k: {"difference": round(d, 6), "ci_low": round(lo, 6),
                "ci_high": round(hi, 6)}
            for k, (d, lo, hi) in comparisons.items()},
        "paired_correctness_by_scope": comparisons_split,
        "same_question_comparison": same_question,
        "refusals": refusal_detail,
        "mh011": mh011,
    }
    write_metrics(METRICS_DIR / "05_judged_arms.json", record)
    print(f"  Saved -> {METRICS_DIR}/05_judged_arms.json")
    print("""
  That file is the measured record. CI has no key by design and cannot
  recompute these numbers, so the memo is verified against this record
  rather than against prose. It is written only by a run that reached
  this point: a keyless run, or one whose judge failed the gate, leaves
  it untouched.
""")

## G. Verdict

In [ ]:
if arm_reports:
    _lines = "\n".join(
        f"   {name:<14} faithfulness {rep.faithfulness:.3f} over {rep.n_answered} answered, "
        f"correctness {rep.correctness:.3f},\n"
        f"   {'':<14} {rep.failure_modes.get('answered_oos', 0)} of "
        f"{sum(1 for q in golden if q['category'] == 'out_of_scope')} out-of-scope answered"
        for name, rep in arm_reports.items())
    arms_verdict = f"""6. THE LLM ARMS ARE JUDGED, BY A JUDGE THAT PASSED ITS GATE.
{_lines}
   These are the first answer-quality readings in the project. They
   carry the judge's measured error rate from Section B, and Phase 6
   still has to check that judge for bias before any difference
   BETWEEN arms is believed."""
    mh011_item = """  - The contradiction case (mh-011) is read in Section F: what each
    judged arm does with two sources that disagree. Phase 6 should
    check whether the judge itself noticed."""
elif has_key:
    arms_verdict = f"""6. THE LLM ARMS WERE NOT JUDGED — THE GATE HELD.
   The judge scored {report.overall_accuracy:.0%} in Section B, below the trust threshold, so
   Section F did not run. No answer-quality claim about the LLM arms
   is made anywhere in this notebook."""
    mh011_item = f"""  - The contradiction case (mh-011). Both refund windows are retrieved
    at depth {DEPTH}. Whether any prompt variant FLAGS the conflict
    rather than silently picking one is still unanswered, and needs a
    judge that passes Section B."""
else:
    arms_verdict = """6. THE LLM ARMS ARE NOT JUDGED HERE.
   No credential, so Section F did not run. Everything scored above is
   the extractive baseline, which nobody proposes to ship."""
    mh011_item = f"""  - The contradiction case (mh-011). Both refund windows are retrieved
    at depth {DEPTH}. Whether any prompt variant FLAGS the conflict
    rather than silently picking one is answered by Section F, which
    needs a credential and a judge that passed Section B."""

print("\n" + "=" * 78)
print("PHASE 5 VERDICT")
print("=" * 78)
print(f"""
1. THE JUDGE IS VALIDATED BEFORE IT IS USED.
   {report.judge} scores {report.overall_accuracy:.0%} on {len(VALIDATION_CASES)} cases with known verdicts;
   trustworthy = {report.is_trustworthy}. Reporting faithfulness from an unvalidated
   judge is reporting a measurement from an uncalibrated instrument,
   and the gate makes that impossible to do by accident here.

2. CONTEXT METRICS ARE COMPUTED, NOT ESTIMATED.
   Ground-truth labels exist, so context precision and recall need no
   model. That makes them exact, free, reproducible, and available even
   with no credential — the harness degrades rather than stops.

3. CONTEXT PRECISION MUST BE READ AGAINST ITS CEILING.
   {report_run.context_precision:.3f} observed against {mean_ceiling:.3f} achievable = {report_run.context_precision / mean_ceiling:.0%} of what is
   structurally possible at depth {DEPTH}. The uncorrected number invites
   the conclusion that retrieval is broken when it is doing close to the
   best the chunking and depth allow.

4. FAILURES ARE ATTRIBUTED, NOT AGGREGATED.
   {gen_fail} generation failures and {ret_fail} retrieval failures among unfaithful
   answers. A single hallucination rate would merge them and send the
   fix to whichever component was guessed.{'''
   Both are zero here because a verbatim copier judged by term overlap
   cannot be unfaithful — the machinery is verified, the finding waits
   on a paraphrasing generator and a semantic judge.''' if gen_fail + ret_fail == 0 else ''}

5. TWO METRIC DEFINITIONS THAT DECIDE WHAT THE NUMBERS MEAN.
   The judge is shown the context in exactly the form the generator saw
   it, article-id tags included. Stripping them makes every citation an
   unverifiable claim, which penalises precisely the prompt variants
   that follow the citation instruction.

   Faithfulness covers answered questions only ({report_run.faithfulness:.3f}), not all of
   them ({report_run.faithfulness_all:.3f}). Refusals are vacuously faithful, so including
   them lets a system that answers nothing report a perfect score.

{arms_verdict}

{'' if has_key else '''NOT MEASURED IN THIS RUN — no credential, so the LEXICAL judge ran and
failed validation as designed. Section D's scores are harness output,
not evidence. The answer-quality numbers that ARE evidence come from
the credentialed run and are in reports/metrics/05_judged_arms.json.

'''}STILL OPEN — carried into Phase 6:

  - Judge bias beyond accuracy: LLM judges favour longer answers and
    show position effects. Worth measuring the correlation between
    answer length and score before trusting cross-variant comparisons.
  - Inter-judge agreement. One judge's opinion is one opinion; two
    judges disagreeing on a question is a signal that question is
    genuinely ambiguous.
{mh011_item}

HANDOFF TO PHASE 6: LLM-as-judge deep dive — judge bias measurement,
inter-judge agreement, and qualitative failure analysis on the cases the
harness flags.
""")